# 01 — Baseline Models: Returns & Direction (with short-interest A/B)

Re-run of the return and direction baselines, now testing the project's first
**non-price feature**: FINRA short interest. The prior version established, on
price features alone:

- A small, consistent cross-sectional **rank** edge on return magnitude (IC ~0.014
  at 5-day, ~0.027 at 20-day), positive every fold.
- **No** usable directional-classification edge (models did not beat the majority
  baseline).

The EDA showed short interest has near-zero *linear* correlation with returns
(~0.005) and only a mild link to volatility (~0.086). But correlation cannot see
the interactions a tree finds, and it averages over the 2022 regime where price
signals inverted — exactly where a non-price feature might diversify. So the
honest test is not the heatmap; it is this: **does adding short interest lift the
walk-forward IC / net spread, with vs without?**

## Design

- **Scope to 2018+** (`SHORT_INTEREST_START`). Short interest is ~48% NaN before
  2018; running the A/B on the covered window makes it a fair comparison and stops
  a tree reading the pre-2018 NaN block as a date proxy.
- **A/B feature sets:** `base` (price + macro, stationary) vs `base + short
  interest`. Identical folds, models, and rows — only the feature columns differ.
- **Judged on rank IC and the cost-aware long-short net spread**, plus a 2022-only
  view, since diversifying that regime was the whole reason to add short interest.
- Walk-forward, purge gap, train-only scaling, naive baseline — unchanged.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, XGBClassifier

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
warnings.filterwarnings("ignore")

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
for parent in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (parent / 'src').is_dir():
        PROJECT_ROOT = parent
        break
else:
    raise RuntimeError('Run from inside the StockForecastRisk repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.forecast_engine.features.schema import (
    FEATURE_NAMES, NON_FEATURE_COLUMNS,
    SHORT_INTEREST_FEATURE_NAMES, SHORT_INTEREST_START,
)
from src.forecast_engine.data.loader import load_processed_features

data = load_processed_features()
data["date"] = pd.to_datetime(data["date"])
print(f'Loaded {len(data):,} rows, {data["symbol"].nunique()} tickers, '
      f'{data["date"].min().date()} -> {data["date"].max().date()}')

## Scope to the short-interest window and build the two feature sets

`base` = stationary price/macro features, excluding raw price levels and the
short-interest columns. `with_si` = base + the four short-interest features. Both
run on rows from `SHORT_INTEREST_START` onward, where short interest exists.

In [ ]:
NON_STATIONARY_LEVELS = ["sma_10", "sma_20", "sma_50", "sma_200", "ema_12", "ema_26", "vwap_20"]
EXCLUDE = set(NON_FEATURE_COLUMNS) | set(NON_STATIONARY_LEVELS) | {"short_history", "history_rows"}
SI = list(SHORT_INTEREST_FEATURE_NAMES)

base_features = [c for c in FEATURE_NAMES
                 if c not in EXCLUDE and c not in SI and c in data.columns]
si_features = [c for c in SI if c in data.columns]

RETURN_TARGET, DIRECTION_TARGET = "future_return", "future_direction"

# Scope to the short-interest coverage window for a fair A/B.
scoped = data[data["date"] >= pd.Timestamp(SHORT_INTEREST_START)].copy()

# Drop rows missing any base feature or the targets. For the with_si arm we also
# require the short-interest columns present; within 2018+ they should be dense.
model_data = scoped.dropna(subset=base_features + si_features + [RETURN_TARGET, DIRECTION_TARGET]).reset_index(drop=True)
model_data[DIRECTION_TARGET] = model_data[DIRECTION_TARGET].astype(int)

print(f"scoped to {SHORT_INTEREST_START}+: {len(model_data):,} rows")
print(f"base features : {len(base_features)}")
print(f"+ short interest: {len(si_features)} -> {si_features}")
print(f"up-day base rate: {model_data[DIRECTION_TARGET].mean():.4f}")

## Walk-forward splits (purged)

In [ ]:
N_SPLITS, PURGE_DAYS = 5, 5

def walk_forward_splits(frame, n_splits=N_SPLITS, purge_days=PURGE_DAYS):
    dates = np.sort(frame["date"].unique())
    edges = np.array_split(dates, n_splits + 1)
    for f in range(n_splits):
        train_end = edges[f][-1]
        test_dates = edges[f + 1]
        cutoff = train_end - pd.Timedelta(days=purge_days)
        tr = frame.index[frame["date"] <= cutoff]
        te = frame.index[frame["date"].isin(test_dates)]
        if len(tr) and len(te):
            yield tr, te

folds = list(walk_forward_splits(model_data))
for i, (tr, te) in enumerate(folds, 1):
    print(f"fold {i}: train {len(tr):>6,} (to {model_data.loc[tr,'date'].max().date()}) | "
          f"test {len(te):>5,} ({model_data.loc[te,'date'].min().date()}->{model_data.loc[te,'date'].max().date()})")

## Metrics and the cost-aware long-short spread

In [ ]:
def daily_rank_ic(dates, y_true, y_pred):
    df = pd.DataFrame({"d": np.asarray(dates), "y": np.asarray(y_true), "p": np.asarray(y_pred)})
    vals = []
    for _, g in df.groupby("d"):
        if g["y"].nunique() > 2 and g["p"].nunique() > 2:
            c = spearmanr(g["p"], g["y"]).correlation
            if np.isfinite(c): vals.append(c)
    vals = np.array(vals)
    return (vals.mean() if vals.size else np.nan), vals

DECILE, COST_BPS = 0.10, 5.0
def long_short_net(dates, y_true, y_pred, decile=DECILE, cost_bps=COST_BPS):
    df = pd.DataFrame({"d": np.asarray(dates), "y": np.asarray(y_true), "p": np.asarray(y_pred)})
    nets = []
    cost = cost_bps / 1e4
    for _, g in df.groupby("d"):
        if len(g) < 20: continue
        hi = g["p"] >= g["p"].quantile(1 - decile)
        lo = g["p"] <= g["p"].quantile(decile)
        if hi.sum() and lo.sum():
            nets.append((g.loc[hi,"y"].mean() - g.loc[lo,"y"].mean()) - 4*cost)
    return float(np.mean(nets)) if nets else np.nan

def make_reg(): return XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42, verbosity=0, tree_method="hist")
def make_clf(): return XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42, verbosity=0,
    eval_metric="logloss", tree_method="hist")

## Arm 1 — Return magnitude: base vs base + short interest

Ridge and XGBoost, each run on both feature sets. The question is whether the
`with_si` columns lift the rank IC or the net spread over `base`.

In [ ]:
# Pre-materialize arrays once (memory-safe).
X_base = model_data[base_features].to_numpy(np.float32)
X_si   = model_data[base_features + si_features].to_numpy(np.float32)
y_ret  = model_data[RETURN_TARGET].to_numpy(np.float32)
dates_all = model_data["date"].to_numpy()

def run_return_arm(X, label):
    rows = []
    for i, (tr, te) in enumerate(folds, 1):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y_ret[tr], y_ret[te]
        dte = dates_all[te]
        sc = StandardScaler().fit(Xtr)
        rp = Ridge(alpha=1.0).fit(sc.transform(Xtr), ytr).predict(sc.transform(Xte))
        xp = make_reg().fit(Xtr, ytr).predict(Xte)
        for name, pred in [("ridge", rp), ("xgboost", xp)]:
            ic, _ = daily_rank_ic(dte, yte, pred)
            rows.append({"fold": i, "features": label, "model": name,
                         "ic": ic, "net": long_short_net(dte, yte, pred)})
    return pd.DataFrame(rows)

ret = pd.concat([run_return_arm(X_base, "base"),
                 run_return_arm(X_si, "base+SI")], ignore_index=True)
ret_summary = ret.groupby(["features", "model"])[["ic", "net"]].mean()
print("Return arm — base vs base+short interest (mean across folds):")
display(ret_summary.round(5))

In [ ]:
# The A/B verdict: does short interest lift IC / net spread?
piv_ic = ret.groupby(["features","model"])["ic"].mean().unstack("features")
piv_net = ret.groupby(["features","model"])["net"].mean().unstack("features")
piv_ic["IC lift"] = piv_ic["base+SI"] - piv_ic["base"]
piv_net["net lift"] = piv_net["base+SI"] - piv_net["base"]
print("Rank IC:"); display(piv_ic.round(5))
print("Net long-short spread:"); display(piv_net.round(6))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, metric, title in [(axes[0], "ic", "Rank IC by fold"), (axes[1], "net", "Net spread by fold")]:
    for feats, style in [("base","-o"), ("base+SI","--s")]:
        for model, color in [("ridge","tab:blue"), ("xgboost","tab:orange")]:
            sub = ret[(ret.features==feats)&(ret.model==model)].sort_values("fold")
            ax.plot(sub.fold, sub[metric], style, color=color, alpha=0.8,
                    label=f"{model} {feats}")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_title(title); ax.set_xlabel("fold")
axes[0].legend(fontsize=7); axes[1].axhline(0, color="red", ls="--")
plt.tight_layout(); plt.show()

## Arm 2 — Direction: base vs base + short interest

Accuracy vs the majority base rate, and AUC vs 0.5, for both feature sets.

In [ ]:
y_dir = model_data[DIRECTION_TARGET].to_numpy()
base_rate = model_data[DIRECTION_TARGET].mean()

def run_dir_arm(X, label):
    rows = []
    for i, (tr, te) in enumerate(folds, 1):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y_dir[tr], y_dir[te]
        sc = StandardScaler().fit(Xtr)
        lp = LogisticRegression(max_iter=1000).fit(sc.transform(Xtr), ytr).predict_proba(sc.transform(Xte))[:,1]
        xp = make_clf().fit(Xtr, ytr).predict_proba(Xte)[:,1]
        for name, proba in [("logistic", lp), ("xgboost", xp)]:
            auc = roc_auc_score(yte, proba) if len(np.unique(yte))>1 else np.nan
            acc = accuracy_score(yte, (proba>=0.5).astype(int))
            rows.append({"fold": i, "features": label, "model": name, "accuracy": acc, "auc": auc})
    return pd.DataFrame(rows)

dir_res = pd.concat([run_dir_arm(X_base, "base"),
                     run_dir_arm(X_si, "base+SI")], ignore_index=True)
print(f"majority base rate: {base_rate:.4f}")
display(dir_res.groupby(["features","model"])[["accuracy","auc"]].mean().round(5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, metric, ref, reflabel in [(axes[0], "accuracy", max(base_rate,1-base_rate), "base rate"),
                                   (axes[1], "auc", 0.5, "random")]:
    for feats, style in [("base","-o"),("base+SI","--s")]:
        for model, color in [("logistic","tab:blue"),("xgboost","tab:orange")]:
            sub = dir_res[(dir_res.features==feats)&(dir_res.model==model)].sort_values("fold")
            ax.plot(sub.fold, sub[metric], style, color=color, alpha=0.8, label=f"{model} {feats}")
    ax.axhline(ref, color="red", ls="--", label=reflabel)
    ax.set_title(f"Direction {metric} by fold"); ax.set_xlabel("fold")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

## The 2022 regime — the reason short interest was added

Full-sample metrics average over regimes. Short interest's hoped-for value was
*diversifying the 2022 momentum crash*, where price signals inverted. This trains
through 2021 and tests on 2022 specifically, base vs base+SI — the sharpest test
of whether the non-price feature helps where price alone failed.

In [ ]:
train_mask = model_data["date"] < "2022-01-01"
test_mask = (model_data["date"] >= "2022-01-01") & (model_data["date"] < "2023-01-01")
tr_idx = model_data.index[train_mask]; te_idx = model_data.index[test_mask]

print(f"train <2022: {len(tr_idx):,} rows | test 2022: {len(te_idx):,} rows")
rows = []
for X, label in [(X_base,"base"), (X_si,"base+SI")]:
    Xtr, Xte = X[tr_idx], X[te_idx]
    ytr, yte = y_ret[tr_idx], y_ret[te_idx]
    dte = dates_all[te_idx]
    xp = make_reg().fit(Xtr, ytr).predict(Xte)
    ic, _ = daily_rank_ic(dte, yte, xp)
    rows.append({"features": label, "ic_2022": ic, "net_2022": long_short_net(dte, yte, xp)})
res_2022 = pd.DataFrame(rows).set_index("features")
res_2022.loc["lift"] = res_2022.loc["base+SI"] - res_2022.loc["base"]
print("XGBoost, 2022 out-of-sample:")
display(res_2022.round(5))

## Conclusion — did short interest earn its place?

Fill from the numbers above.

- **Return IC lift (base+SI − base):** ____ — positive and consistent, or noise?
- **Net spread lift:** ____ — does short interest add tradeable edge after cost?
- **Direction:** any lift in accuracy/AUC? (Prior: direction was unforecastable;
  short interest is unlikely to change that, but check.)
- **2022 specifically:** did short interest help in the regime it was meant to
  diversify? This matters more than the full-sample average.

### The decision

- **If short interest lifts IC/net spread (especially in 2022)** → the first
  non-price feature earns its place; keep it, and sourcing more non-price data
  (fundamentals, analyst revisions) is justified.
- **If no lift anywhere, including 2022** → short interest, like the price
  features, does not crack 5-day returns. A clean, rigorous negative: it says the
  return ceiling is not about *this* non-price feature, and points either to
  richer non-price data (fundamentals) or to accepting returns as a thin tilt and
  making the **volatility/risk engine** the deliverable — which the validated vol
  model (IC 0.48, beats GARCH) already supports.

Either outcome is decision-useful: it turns "should I buy fundamentals data?" into
an evidence-based call rather than a guess.